# Пояснение по модели (по-русски, очень простыми словами)

Этот ноутбук — **для чтения и понимания**. Он объясняет, что делалось в `EDA.ipynb`, почему выбраны такие параметры, и как читать метрики/графики.

Важно:
- Ты запускаешь всё **на своём Mac**.
- Если видишь `Num GPUs Available: 0`, значит TensorFlow **не использует видеокарту**, и обучение идёт на **процессоре (CPU)**.


## 1) Какая задача?

Мы решаем задачу **классификации картинок**:
- На вход: маленькая картинка 32×32 (CIFAR-10).
- На выход: один из 10 классов (самолёт, машина, птица, …).

Модель учится отвечать на вопрос: **"что изображено на картинке?"**


## 2) Откуда данные?

CIFAR-10 — это готовый датасет.

Почему мы загружаем его «локально»?
- Потому что в этой среде может не быть интернета.
- Мы положили архив датасета в `.keras/datasets/cifar-10-python.tar.gz`.


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Чтобы не было предупреждений про кэш matplotlib
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('artifacts/mplconfig'))

# Берём локальный загрузчик CIFAR-10 (без интернета)
sys.path.insert(0, 'scripts')
from local_cifar10 import load_cifar10_from_tar

print('TensorFlow:', tf.__version__)
print('GPU count  :', len(tf.config.list_physical_devices('GPU')))

data = load_cifar10_from_tar('.keras/datasets/cifar-10-python.tar.gz')
x_train, y_train = data.x_train, data.y_train
x_test, y_test = data.x_test, data.y_test

print('x_train:', x_train.shape, x_train.dtype)
print('y_train:', y_train.shape, y_train.dtype)


## 3) Почему мы используем ResNet50?

Представь, что ResNet50 — это **очень умный «глаз»**, который уже научился видеть формы/текстуры на миллионах картинок (ImageNet).

Мы делаем так:
- Берём этот «глаз» (ResNet50).
- Сверху добавляем **простую "голову"** (несколько слоёв), которая учится отличать 10 классов CIFAR-10.

Это называется **transfer learning**: «перенос знаний».


## 4) Почему нужно `preprocess_input`?

ResNet50 обучалась на картинках, которые были подготовлены определённым способом.
Если мы подадим «сырые» пиксели (0…255), модель будет хуже понимать данные.

`preprocess_input` — это как **правильный формат** для входа ResNet50.


## 5) Что такое Stage 1 и Stage 2?

### Stage 1 (Head-only)
- **Замораживаем** ResNet50 (её веса не меняются).
- Учим только «голову» сверху.

Зачем так?
- Быстрее.
- Меньше риск «сломать» полезные знания ResNet50.

### Stage 2 (Fine-tuning)
- Немного «размораживаем» часть ResNet50 и дообучаем.
- Очень важное правило: **маленький learning rate**.

На CPU полный fine-tuning очень долгий, поэтому в рабочем ноутбуке мы размораживаем **только последние 20 слоёв**.


## 6) Что означают метрики и графики?

### Accuracy (точность)
Сколько картинок из 100 модель угадала правильно.

### Loss (ошибка)
Насколько сильно модель «ошибается» в среднем.
- Обычно хочется, чтобы **loss уменьшался**.

### Train vs Validation
- `train_*`: как модель учится на тренировочных данных.
- `val_*`: проверка на отложенной части тренировочных данных (модель их не видит при обучении в этом шаге).

Если `train_accuracy` растёт, а `val_accuracy` стоит или падает — это признак **переобучения** (модель запоминает тренировки).


## 7) Итоги обучения (берём из сохранённого файла)

Рабочий ноутбук сохраняет метрики в `artifacts/run_summary.json`.
Давай прочитаем их и объясним простыми словами.


In [ ]:
import json

with open('artifacts/run_summary.json', 'r') as f:
    s = json.load(f)

print('TensorFlow:', s['tensorflow'])
print('GPUs      :', s['gpus'])

print('\nStage 1 (учили только "голову")')
print('  Лучшая val_accuracy:', round(s['stage1']['val_acc_best'], 4))
print('  Лучшая val_loss    :', round(s['stage1']['val_loss_best'], 4))
print('  Эпох прошло        :', s['stage1']['epochs_ran'])

print('\nStage 2 (дообучали последние 20 слоёв)')
print('  Лучшая val_accuracy:', round(s['stage2']['val_acc_best'], 4))
print('  Лучшая val_loss    :', round(s['stage2']['val_loss_best'], 4))
print('  Эпох прошло        :', s['stage2']['epochs_ran'])

print('\nТест (самая честная проверка)')
print('  test_accuracy:', round(s['test_accuracy'], 4))
print('  test_loss    :', round(s['test_loss'], 4))


## 8) Как интерпретировать результат (по‑человечески)

- Если `val_accuracy` чуть выросла на Stage 2 — значит «тонкая настройка» помогла.
- Но если выросла на validation, а на `test_accuracy` не выросла — значит улучшение могло быть случайным или модель переобучилась.

Главное правило:
- **Тест** — это самая важная цифра для честного сравнения.
